In [17]:
class GaussianMixture(object):

    def __init__(self, k=3, max_iter=100, eps=1e-4, reg_cov=1e-6, random_state=None):
        self.k = k
        self.max_iter = max_iter
        self.eps = eps
        self.reg_cov = reg_cov
        self.random_state = random_state

        # Parameters to be estimated
        self.phi_ = None    # shape (k,)
        self.mu_ = None     # shape (k, n_features)
        self.sigma_ = None  # shape (k, n_features, n_features)

    def _log_gaussian(self, X, mu, sigma):
        """
        Compute log N(x | mu, sigma) for every row in X.
        """
        n_features = X.shape[1]
        diff = X - mu

        sign, log_det = np.linalg.slogdet(sigma)
        if sign <= 0:
            raise ValueError("Covariance matrix sigma must be positive definite.")

        # Solve sigma * z = diff.T instead of computing explicit inverse
        solved = np.linalg.solve(sigma, diff.T).T
        mahalanobis = np.sum(diff * solved, axis=1)

        return -0.5 * (n_features * np.log(2.0 * np.pi) + log_det + mahalanobis)

    @staticmethod
    def _logsumexp(A, axis=1):
        """Numerically stable log(sum(exp(A)))."""
        max_A = np.max(A, axis=axis, keepdims=True)
        result = max_A + np.log(np.sum(np.exp(A - max_A), axis=axis, keepdims=True))
        return np.squeeze(result, axis=axis)

    def _estimate_log_prob(self, X):
        """Log joint probability: log(phi_j * N(x | mu_j, sigma_j))."""
        log_prob = np.empty((X.shape[0], self.k))
        for j in range(self.k):
            log_prob[:, j] = np.log(self.phi_[j]) + self._log_gaussian(X, self.mu_[j], self.sigma_[j])
        return log_prob

    def _e_step(self, X):
        """
        E-step: calculate posterior responsibilities:
        w_j^(i) := p(z^(i) = j | x^(i); phi, mu, sigma)
        """
        log_joint = self._estimate_log_prob(X)
        log_norm = self._logsumexp(log_joint, axis=1)

        log_w = log_joint - log_norm[:, None]
        w = np.exp(log_w)  # w_j^(i)
        avg_log_likelihood = np.mean(log_norm)

        return w, avg_log_likelihood

    def _m_step(self, X, w):
        """
        M-step: update parameters phi, mu, and sigma based on weights w.
        """
        n_samples, n_features = X.shape

        sum_w = w.sum(axis=0) + 1e-10

        # phi_j := 1/m * sum_i w_j^(i)
        self.phi_ = sum_w / n_samples

        # mu_j := sum_i (w_j^(i) * x^(i)) / sum_i w_j^(i)
        self.mu_ = (w.T @ X) / sum_w[:, None]

        # sigma_j := sum_i w_j^(i) (x^(i) - mu_j)(x^(i) - mu_j)^T / sum_i w_j^(i)
        self.sigma_ = np.empty((self.k, n_features, n_features))
        for j in range(self.k):
            diff = X - self.mu_[j]
            cov = ((diff * w[:, j:j+1]).T @ diff) / sum_w[j]
            # Regularization to prevent singular covariance
            cov += self.reg_cov * np.eye(n_features)
            self.sigma_[j] = cov

    def fit(self, X):
        """
        Fit the Gaussian Mixture Model to data X via EM algorithm.
        """
        X = np.asarray(X, dtype=float)

        if X.ndim != 2 or X.shape[0] == 0:
            raise ValueError("X must be a non-empty 2D array.")
        if not np.all(np.isfinite(X)):
            raise ValueError("X must contain only finite values.")
        if not 1 <= self.k <= X.shape[0]:
            raise ValueError("k must be between 1 and n_samples.")

        rng = np.random.default_rng(self.random_state)
        n_samples, n_features = X.shape

        # Initialize mu from randomly selected data points
        indices = rng.choice(n_samples, size=self.k, replace=False)
        self.mu_ = X[indices].copy()

        # Initialize uniform phi: phi_j = 1/k
        self.phi_ = np.full(self.k, 1.0 / self.k)

        # Initialize shared sigma from data covariance
        global_cov = np.atleast_2d(np.cov(X, rowvar=False)) + self.reg_cov * np.eye(n_features)
        self.sigma_ = np.repeat(global_cov[None, :, :], self.k, axis=0)

        self.lower_bound_ = -np.inf
        self.converged_ = False
        self.n_iter_ = 0
        self.log_likelihood_history_ = []

        # EM iterations
        for i in range(self.max_iter):
            # E-step
            w, log_likelihood = self._e_step(X)
            self.log_likelihood_history_.append(log_likelihood)

            # Check convergence
            if i > 0:
                improvement = log_likelihood - self.lower_bound_
                if abs(improvement) < self.eps:
                    self.converged_ = True
                    self.n_iter_ = i
                    break

            self.lower_bound_ = log_likelihood

            # M-step
            self._m_step(X, w)
            self.n_iter_ = i + 1

        # Final likelihood evaluation
        w, self.lower_bound_ = self._e_step(X)
        self.log_likelihood_history_.append(self.lower_bound_)

        return self

    def predict_proba(self, X):
        """Posterior probabilities w_j^(i) for each sample."""
        X = np.asarray(X, dtype=float)
        w, _ = self._e_step(X)
        return w

    def predict(self, X):
        """Return the most likely component (cluster assignment) for each sample."""
        return np.argmax(self.predict_proba(X), axis=1)

    def score_samples(self, X):
        """Return log p(x) for each sample."""
        X = np.asarray(X, dtype=float)
        return self._logsumexp(self._estimate_log_prob(X), axis=1)

    def score(self, X):
        """Return average log-likelihood across all samples."""
        return np.mean(self.score_samples(X))

In [18]:
import os
import h5py
import time
import matplotlib
import numpy as np
import pandas as pd
import seaborn as sns
from pandas import DataFrame
import matplotlib.pyplot as plt
from matplotlib import gridspec
%matplotlib inline

In [19]:
filenames = [
    "N-CMAPSS_DS01-005.h5",
    "N-CMAPSS_DS02-006.h5",
    "N-CMAPSS_DS03-012.h5",
    "N-CMAPSS_DS04.h5",
    "N-CMAPSS_DS05.h5",
    "N-CMAPSS_DS06.h5",
    "N-CMAPSS_DS07.h5",
    "N-CMAPSS_DS08a-009.h5",
    "N-CMAPSS_DS08c-008.h5"
]

In [20]:
### Set-up - Define file location
filename = "Data/N-CMAPSS_DS08a-009.h5"

In [21]:
# Time tracking, Operation time (min):  0.003
t = time.process_time()  

# Load data
with h5py.File(filename, 'r') as hdf:
        # Development set
        W_dev = np.array(hdf.get('W_dev'))             # W
        X_s_dev = np.array(hdf.get('X_s_dev'))         # X_s
        X_v_dev = np.array(hdf.get('X_v_dev'))         # X_v
        T_dev = np.array(hdf.get('T_dev'))             # T
        Y_dev = np.array(hdf.get('Y_dev'))             # RUL  
        A_dev = np.array(hdf.get('A_dev'))             # Auxiliary

        # Test set
        W_test = np.array(hdf.get('W_test'))           # W
        X_s_test = np.array(hdf.get('X_s_test'))       # X_s
        X_v_test = np.array(hdf.get('X_v_test'))       # X_v
        T_test = np.array(hdf.get('T_test'))           # T
        Y_test = np.array(hdf.get('Y_test'))           # RUL  
        A_test = np.array(hdf.get('A_test'))           # Auxiliary
        
        # Varnams
        W_var = np.array(hdf.get('W_var'))
        X_s_var = np.array(hdf.get('X_s_var'))  
        X_v_var = np.array(hdf.get('X_v_var')) 
        T_var = np.array(hdf.get('T_var'))
        A_var = np.array(hdf.get('A_var'))
        
        # from np.array to list dtype U4/U5
        W_var = list(np.array(W_var, dtype='U20'))
        X_s_var = list(np.array(X_s_var, dtype='U20'))  
        X_v_var = list(np.array(X_v_var, dtype='U20')) 
        T_var = list(np.array(T_var, dtype='U20'))
        A_var = list(np.array(A_var, dtype='U20'))
                          
W = np.concatenate((W_dev, W_test), axis=0)  
X_s = np.concatenate((X_s_dev, X_s_test), axis=0)
X_v = np.concatenate((X_v_dev, X_v_test), axis=0)
T = np.concatenate((T_dev, T_test), axis=0)
Y = np.concatenate((Y_dev, Y_test), axis=0) 
A = np.concatenate((A_dev, A_test), axis=0) 
    
print('')
print("Operation time (min): " , (time.process_time()-t)/60)
print('')
print ("W shape: " + str(W_dev.shape))
print ("X_s shape: " + str(X_s_dev.shape))
print ("X_v shape: " + str(X_v_dev.shape))
print ("T shape: " + str(T_dev.shape))
print ("A shape: " + str(A_dev.shape))


Operation time (min):  0.0171273838833334

W shape: (4885389, 4)
X_s shape: (4885389, 14)
X_v shape: (4885389, 14)
T shape: (4885389, 10)
A shape: (4885389, 4)


In [22]:
model = GaussianMixture(k=6)

In [ ]:
model.fit(X_s_dev)

In [ ]:
print(model.mu_)